In [54]:
# from google.colab import drive
# drive.mount('/content/drive')
# %cd "/content/drive/MyDrive/"

import os
os.chdir(r"C:\Z")  # Cambia el directorio de trabajo
print(os.getcwd())  # Verifica que cambió correctamente el directorio base

C:\Z


In [55]:
from SciServer import CasJobs as cj
import pandas as pd
import numpy as np
import requests
import time
import pickle
import pandas as pd
from concurrent.futures import ThreadPoolExecutor, as_completed
pd.set_option('display.max_rows', 20)

# Iniciar sesión
from SciServer import Authentication
Authentication.login(UserName = "Alberto_2002", Password = "#Blackie2002")

'23d67a551da4406f90222f5c1e8987c1'

In [ ]:
# Parámetros iniciales
batch_size_inicial = 75000       # tamaño inicial del lote
min_batch_size = 50000           # tamaño mínimo de lote permitido
max_retries = 10                 # máximo número de reintentos por lote
offset = 0                       # registro inicial a recuperar, dejamos la variable por si hace falta aunque por ahora no hace mucho
all_results = []
batch_results = []

while offset <= 100000:

    with open('TFGF/Python/extra/redshift_init.pickle', 'rb') as handle:
        redshift_init = pickle.load(handle)

    redshift_init = float(str(redshift_init)[:9]) # Solo se toman los primeros 7 digitos porque si no se excede el límite de la consulta
    current_batch_size = batch_size_inicial
    retries = 0
    success = False
    iteration_start = time.time()  # marca de tiempo para controlar el rate limit

    # Intentar ejecutar el query con el tamaño de lote actual. Si hay error, se reduce el lote y se reintenta.             2.121375e-07
    while not success and retries < max_retries:
        sql_query = f"""
        SELECT
            p.objid,
            s.specObjID,
            dbo.fGetUrlFitsSpectrum(s.specObjID) AS fits_url,
            s.z AS redshift
        FROM 
            PhotoObj AS p
        JOIN 
            SpecObj AS s ON s.bestobjid = p.objid
        WHERE 
            s.z BETWEEN {redshift_init} AND 9
            AND s.zWarning = 0
            AND (s.class = 'QSO' OR s.class = 'GALAXY')
        ORDER BY s.z
        OFFSET {offset} ROWS
        FETCH NEXT {current_batch_size} ROWS ONLY
        """
        try:
            batch_results = pd.DataFrame(cj.executeQuery(sql_query, context="DR18"))
            success = True
            all_results.append(batch_results)
            offset += len(batch_results)

            with open('TFGF/Python/extra/all_results.pickle', 'wb') as handle:
                pickle.dump(all_results, handle)
            print(f"Recuperados {offset} registros hasta ahora.")

            ultimo_redshift = all_results[len(all_results)-2]["redshift"].iloc[0]

            if float(str(ultimo_redshift)[:9]) == redshift_init:    # Si el redshift no cambió, se incrementa un poco para evitar loops infinitos
                ultimo_redshift = ultimo_redshift + 0.0000001

            with open('TFGF/Python/extra/redshift_init.pickle', 'wb') as handle:
                pickle.dump(ultimo_redshift, handle)

        except Exception as e:
            print(f"Error en la consulta con batch_size = {current_batch_size}: {e}")
            # Reducir el tamaño del lote, pero sin bajar de min_batch_size
            current_batch_size = max(min_batch_size, current_batch_size // 2)
            retries += 1
            time.sleep(2)  # Esperar un poco antes de reintentar

    if len(batch_results) == 0:
        print("No hay más registros para recuperar.")
        break

    if retries == max_retries:
        print("No se pudo recuperar el batch tras varios reintentos. Terminando.")
        break

1.000022


In [ ]:
with open('TFGF/Python/extra/all_results.pickle', 'rb') as handle:
    all_results = pickle.load(handle)

# Combinar todos los resultados (suponiendo que cada batch es un DataFrame)
combined_results = pd.concat(all_results, ignore_index=True)
print(f"Total de registros recuperados: {len(combined_results)}")

# Extraer las URLs de los archivos FITS
fits_urls = combined_results["fits_url"].tolist()
print(f"Se han obtenido {len(fits_urls)} URLs de FITS.")

ultimo_redshift = all_results[len(all_results)-2]["redshift"].iloc[0]
print(f"Último redshift recuperado: {ultimo_redshift:.8}")

with open('TFGF/Python/extra/redshift_init.pickle', 'wb') as handle:
    pickle.dump(ultimo_redshift, handle)

Total de registros recuperados: 50000
Se han obtenido 50000 URLs de FITS.
Último redshift recuperado: 1.000022


In [ ]:
import os

# Directorio de descarga
output_dir = r"C:/Z/TFGF NO DRIVE/Python/spectrums"
os.makedirs(output_dir, exist_ok=True)

archivos_descargados = []

def download_file(url):
    filename = os.path.basename(url)
    local_path = os.path.join(output_dir, filename)
    
    # Saltar si ya se descargó este archivo
    if os.path.exists(local_path):
        print(f"Saltando {filename} (ya descargado).")
        return filename, None
    
    try:
        response = requests.get(url)
        response.raise_for_status()
        with open(local_path, "wb") as f:
            f.write(response.content)
        print(f"Descargado: {filename}")
        return filename, None
    except Exception as e:
        print(f"Error al descargar {url}: {e}")
        return filename, e

# Número de descargas en paralelo
paralelo = 100
i=0

with ThreadPoolExecutor(max_workers=paralelo) as executor:
    future_to_url = {executor.submit(download_file, url): url for url in fits_urls}
    
    for future in as_completed(future_to_url):
        i+=1
        print(f"Descargando archivo {i} de {len(fits_urls)}")
        filename, error = future.result()
        if error is None:
            archivos_descargados.append(filename)

print("Descarga finalizada")

Saltando spec-8375-57520-0581.fits (ya descargado).
Saltando spec-7840-57003-0970.fits (ya descargado).
Saltando spec-9233-58035-0419.fits (ya descargado).
Saltando spec-5354-55927-0758.fits (ya descargado).
Saltando spec-9433-58131-0771.fits (ya descargado).
Saltando spec-8381-57512-0588.fits (ya descargado).
Saltando spec-9608-58137-0304.fits (ya descargado).
Saltando spec-8238-58171-0226.fits (ya descargado).
Saltando spec-9217-57934-0680.fits (ya descargado).
Saltando spec-8821-57731-0552.fits (ya descargado).
Saltando spec-9444-58073-0361.fits (ya descargado).
Saltando spec-6669-56413-0170.fits (ya descargado).
Saltando spec-11379-58438-0616.fits (ya descargado).
Saltando spec-8186-57452-0194.fits (ya descargado).
Saltando spec-8126-56956-0460.fits (ya descargado).
Saltando spec-7414-56748-0996.fits (ya descargado).
Saltando spec-9149-58039-0044.fits (ya descargado).
Saltando spec-9439-58015-0794.fits (ya descargado).
Saltando spec-9378-58071-0087.fits (ya descargado).
Saltando sp